# 9장 · 가설검정과 p값 ⚖️

웹 챕터 **"가설검정과 p값"** 과 짝을 이루는 Python 실습입니다.
"우연인가 진짜인가"를 시뮬레이션으로 직접 세어 **p값**을 구하고, scipy 로 검산합니다.

준비물: `pip install -r requirements.txt`

> 🤖 **AI 튜터**: 아래 셀을 실행하면 웹앱처럼 화면 오른쪽 아래에 🤖 버튼이 생깁니다.
> 버튼을 누르면 채팅창이 열리고, 어디를 스크롤해도 그 자리에 떠 있어요. 막히면 언제든 질문하세요.
> (로컬 Ollama 필요. 꺼져 있으면 켜는 방법이 안내되고, 튜터 없이도 학습에는 지장 없습니다.)

In [ ]:
# 🤖 AI 튜터 — 실행하면 오른쪽 아래에 🤖 버튼이 떠요 (웹앱과 같은 로컬 AI, 클릭해서 열기)
from tutor import chat

chat('ch9')

## 1. 동전이 치우쳤나? 시뮬레이션으로 p값 구하기

어떤 동전을 100번 던졌더니 앞면이 **60번** 나왔습니다.
귀무가설 $H_0$: "공정한 동전이다(앞면 확률 0.5)".

**p값** = $H_0$ 이 참일 때, 관측만큼(또는 더 극단적으로) 치우칠 확률.
공정한 동전 100번 던지기를 컴퓨터로 **10만 번** 반복해서 직접 세어 봅시다.
"60번 이상 **또는** 40번 이하"가 나온 비율이 곧 p값입니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['font.family'] = 'Malgun Gothic'   # 한글 폰트 (Windows)
plt.rcParams['axes.unicode_minus'] = False      # 마이너스 기호 깨짐 방지

rng = np.random.default_rng(42)

# 공정한 동전(p=0.5)을 100번 던지는 실험을 10만 번 반복
n_sim = 100_000
heads = rng.binomial(n=100, p=0.5, size=n_sim)  # 각 실험에서 나온 앞면 횟수

# 관측(60번)만큼 또는 더 극단적: 60번 이상 또는 40번 이하
extreme = (heads >= 60) | (heads <= 40)
p_sim = extreme.mean()
print(f'극단적으로 치우친 실험: {int(extreme.sum()):,} / {n_sim:,}')
print(f'시뮬레이션 p값 ≈ {p_sim:.4f}')   # 약 0.057

## 2. scipy 로 검산: `binomtest`

시뮬레이션 대신 이항분포(5장!)로 정확하게 계산해 주는 함수가 있습니다.

In [ ]:
from scipy.stats import binomtest

res = binomtest(k=60, n=100, p=0.5)   # 100번 중 60번, 공정하다는 가정
print(f'정확한 p값 = {res.pvalue:.4f}')
print('결론: p ≈ 0.057 ≥ 0.05 → 유의수준 0.05 에서는 기각하지 못함 (증거 부족)')

## 3. 그림으로 보기: "우연의 분포" 위에 관측값 올리기

공정한 동전이 만들어내는 앞면 횟수의 분포(우연의 분포)를 그리고,
관측한 60번이 어디쯤인지 표시해 봅니다. 빨간 선 바깥 영역의 비율이 p값입니다.

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(heads, bins=np.arange(29.5, 71.5, 1), color='#a5b4fc', edgecolor='white')
plt.axvline(60, color='#e11d48', lw=2.5, label='관측: 앞면 60번')
plt.axvline(40, color='#e11d48', lw=2.5, ls='--', label='반대쪽 극단: 40번')
plt.title('공정한 동전 100번 던지기 10만 회: 앞면 횟수의 분포')
plt.xlabel('앞면 횟수'); plt.ylabel('실험 수'); plt.legend()
plt.show()

## 4. A/B 테스트: 10% vs 12%

버튼 A: 1,000명 중 100명 구매(10%) / 버튼 B: 1,000명 중 120명 구매(12%).

귀무가설 세상에서는 두 버튼이 똑같으므로, 모두가 **합동 비율**
$(100+120)/(1000+1000) = 0.11$ 을 공유합니다. 이 세상을 시뮬레이션해서
"차이가 0.02 이상 벌어지는 일"이 얼마나 흔한지 세어 봅니다.

In [ ]:
def ab_pvalue(conv_a, conv_b, n, n_sim=100_000):
    """'차이 없음'(합동 비율) 세상을 시뮬레이션해 두 비율 차이의 p값을 구한다."""
    obs_diff = conv_b / n - conv_a / n            # 관측된 차이
    pooled = (conv_a + conv_b) / (2 * n)          # 합동 비율
    a = rng.binomial(n, pooled, size=n_sim) / n   # 우연 세상의 그룹 A
    b = rng.binomial(n, pooled, size=n_sim) / n   # 우연 세상의 그룹 B
    return float((np.abs(b - a) >= abs(obs_diff) - 1e-12).mean())

p1000 = ab_pvalue(100, 120, 1000)
print(f'n=1,000 → p ≈ {p1000:.3f}  (0.05 보다 훨씬 큼, 증거 부족)')

### 같은 차이, 표본만 5배로

전환율은 그대로 10% vs 12% 인데 그룹당 5,000명이라면?

In [ ]:
p5000 = ab_pvalue(500, 600, 5000)   # 같은 10% vs 12%, 그룹당 5,000명
print(f'n=5,000 → p ≈ {p5000:.4f}  (유의!)')
print('같은 2%p 차이라도 표본이 커지면 우연으로는 설명하기 어려워집니다. 7장의 표준오차 σ/√n.')

## 5. p값의 3대 오해 (꼭 기억!)

1. **p값은 "귀무가설이 참일 확률"이 아니다**: p값은 P(이런 데이터 | H₀ 참).
   방향을 뒤집으면 4장의 P(A|B) ≠ P(B|A) 함정에 빠집니다.
2. **p ≥ 0.05 는 "차이 없음의 증명"이 아니다**: 증거 부족일 뿐. 표본이 작으면
   진짜 차이도 못 잡아냅니다.
3. **유의함 ≠ 중요함**: 표본이 아주 크면 하찮은 차이도 유의해집니다.
   p값과 함께 **효과의 크기**를 항상 확인하세요.

## 정리

- 가설검정 = "차이 없다"($H_0$)를 일단 가정하고, 그 세상에서 내 데이터가 얼마나
  드문 일인지(p값)를 잰다.
- p < 유의수준(관례 0.05) 이면 기각합니다. "우연이라기엔 너무 드물다".
- 동전 60/100 은 p ≈ 0.057 로 아슬아슬하게 기각 실패, A/B 10% vs 12%(n=1,000)는
  p ≈ 0.15 로 증거 부족, 같은 차이로 n=5,000 이면 유의.
- AI 현장에서도 모델 비교·배포 결정은 A/B 테스트(가설검정)로 합니다.

다음은 마지막 챕터인 **상관관계와 회귀, 머신러닝의 시작**입니다.
웹 챕터(`/ch/ch10`)와 `ch10_correlation_regression.ipynb` 로 이어가세요!